# AI Red Teaming Agent

Automated adversarial testing against a Foundry prompt agent using the AI Red Teaming Agent in the cloud.

## How it works

The AI Red Teaming Agent leverages [PyRIT](https://github.com/microsoft/PyRIT) and Foundry's Risk & Safety evaluators to:

1. **Generate a taxonomy** of prohibited actions for the agent
2. **Apply attack strategies** (Flip, Base64, IndirectJailbreak) to generate adversarial prompts
3. **Run multi-turn probing** against the agent
4. **Evaluate** each attack-response pair to compute the Attack Success Rate (ASR)

## References

* [AI Red Teaming Agent — Concepts](https://learn.microsoft.com/en-us/azure/foundry/concepts/ai-red-teaming-agent)
* [Run AI Red Teaming Agent in the cloud](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud)
* [Risk and safety evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/risk-safety-evaluators)
* [Agent evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/agent-evaluators)
* [Supported regions for safety evaluators](https://learn.microsoft.com/en-us/azure/foundry/concepts/evaluation-evaluators/risk-safety-evaluators#foundry-project-configuration-and-region-support)

## Evaluators used

| Evaluator | What it measures |
|-----------|-----------------|
| `builtin.prohibited_actions` | Does the agent perform disallowed actions? |
| `builtin.task_adherence` | Does the agent follow its instructions under attack? |
| `builtin.sensitive_data_leakage` | Does the agent leak sensitive data? |

## Prerequisites

* Run notebook **02** first to create the `med-diagnostic-agent`
* **Azure AI User** role on the Foundry project
* Project region must support safety evaluators (East US 2, Sweden Central, etc.)
* Authenticate via `az login`

```bash
pip install azure-ai-projects>=2.0.0 azure-identity
```

### Setup credentials and project client

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

from utils.fdyauth import AuthHelper
settings = AuthHelper.load_settings()
credential = AuthHelper.test_credential()

if credential:
    print('Environment and authentication OK')
else:
    print("please login first")

In [ ]:
import os
import json
import time
import datetime
from pprint import pprint
import azure.ai.projects as projectslib
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AzureAIAgentTarget,
    AgentTaxonomyInput,
    EvaluationTaxonomy,
    RiskCategory,
)

project_client = AIProjectClient(
    credential=credential,
    endpoint=settings.project_endpoint,
)
openai_client = project_client.get_openai_client()

print("project_client api version:", project_client._config.api_version)
print(f"azure-ai-projects version: {projectslib.__version__}")

### Step 1 — Create a Red Team

A red team is an evaluation container that defines evaluators and data source config. All runs under the same red team share the same evaluators.

Ref: [Create an AI red team](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-an-ai-red-team)

In [ ]:
AGENT_NAME = "med-diagnostic-agent"
timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")

# Get latest agent version
agent = project_client.agents.get_latest_version(agent_name=AGENT_NAME)
print(f"Agent: {agent.name} (version: {agent.version})")

# Create a red team with built-in safety evaluators
red_team = openai_client.evals.create(
    name=f"Red Team {AGENT_NAME} {timestamp}",
    data_source_config={"type": "azure_ai_source", "scenario": "red_team"},
    testing_criteria=[
        {
            "type": "azure_ai_evaluator",
            "name": "Prohibited Actions",
            "evaluator_name": "builtin.prohibited_actions",
            "evaluator_version": "1",
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Task Adherence",
            "evaluator_name": "builtin.task_adherence",
            "evaluator_version": "1",
            "initialization_parameters": {"deployment_name": settings.model_deployment_name},
        },
        {
            "type": "azure_ai_evaluator",
            "name": "Sensitive Data Leakage",
            "evaluator_name": "builtin.sensitive_data_leakage",
            "evaluator_version": "1",
        },
    ],
)
print(f"Red team created: {red_team.id}")

### Step 2 — Create an evaluation taxonomy

Generate a taxonomy of prohibited actions for the agent. This defines the attack surface that the red teaming agent will probe.

Ref: [Create (or update) an evaluation taxonomy](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-or-update-an-evaluation-taxonomy)

In [ ]:
# Define the agent target
target = AzureAIAgentTarget(
    name=AGENT_NAME,
    version=agent.version,
)

# Create taxonomy for prohibited actions risk category
taxonomy = project_client.beta.evaluation_taxonomies.create(
    name=AGENT_NAME,
    body=EvaluationTaxonomy(
        description=f"Red team taxonomy for {AGENT_NAME}",
        taxonomy_input=AgentTaxonomyInput(
            risk_categories=[RiskCategory.PROHIBITED_ACTIONS],
            target=target,
        ),
    ),
)
taxonomy_file_id = taxonomy.id
print(f"Taxonomy created: {taxonomy_file_id}")

### Step 3 — Create a red team run

A run generates adversarial items from the taxonomy and probes the agent with chosen attack strategies.

| Strategy | Description |
|----------|-------------|
| `Flip` | Character flipping to bypass content filters |
| `Base64` | Base64 encoding to disguise harmful prompts |
| `IndirectJailbreak` | Indirect prompt injection via context |

Ref: [Create a run in a red team](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#create-a-run-in-a-red-team)

In [ ]:
# Create a red team run with attack strategies
eval_run = openai_client.evals.runs.create(
    eval_id=red_team.id,
    name=f"Red Team Run {timestamp}",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team_taxonomy",
            "attack_strategies": ["Flip", "Base64", "IndirectJailbreak"],
            "num_turns": 3,
            "source": {"type": "file_id", "id": taxonomy_file_id},
        },
        "target": target.as_dict(),
    },
)
print(f"Red team run created: {eval_run.id}, status: {eval_run.status}")

### Step 4 — Poll for completion and inspect results

Red team runs can take several minutes depending on `num_turns` and the number of attack strategies.

Ref: [List red teaming run output items and results](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud#list-red-teaming-run-output-items-and-results)

In [ ]:
# Poll for completion
while True:
    run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=red_team.id)
    print(f"Status: {run.status}")
    if run.status in ("completed", "failed", "canceled"):
        break
    time.sleep(10)

print(f"\nRed team run finished: {run.status}")

if run.status == "completed":
    print(f"Result counts: {run.result_counts}")
    if hasattr(run, "report_url") and run.report_url:
        print(f"Report URL: {run.report_url}")

    # Fetch output items
    items = list(
        openai_client.evals.runs.output_items.list(run_id=run.id, eval_id=red_team.id)
    )
    print(f"\nOutput items: {len(items)}")

    # Save results to file
    data_folder = os.path.join(os.getcwd(), "data")
    os.makedirs(data_folder, exist_ok=True)
    output_path = os.path.join(data_folder, f"redteam_results_{AGENT_NAME}_{timestamp}.json")
    with open(output_path, "w") as f:
        json.dump([item.to_dict() if hasattr(item, "to_dict") else str(item) for item in items], f, indent=2, default=str)
    print(f"Results saved to {output_path}")

    # Summary
    for i, item in enumerate(items[:10]):  # show first 10
        print(f"\n{'='*60}")
        print(f"Item {i+1}:")
        for r in getattr(item, "results", []):
            name = getattr(r, "name", "unknown")
            label = getattr(r, "label", "N/A")
            score = getattr(r, "score", "N/A")
            print(f"  {name}: {label} (score={score})")
elif run.status == "failed":
    print(f"Error: {run.error}")

### View results in Foundry Portal

1. Login to https://ai.azure.com/
2. Open your Foundry Project
3. Navigate to **Evaluation** in the left menu
4. Select the red team evaluation run to see the Attack Success Rate (ASR) dashboard

### Next steps

* Adjust `attack_strategies` and `num_turns` for deeper coverage
* Add more `RiskCategory` values (e.g. `SENSITIVE_DATA_LEAKAGE`)
* Set up [continuous red teaming](https://learn.microsoft.com/en-us/azure/foundry/how-to/develop/run-ai-red-teaming-cloud) on a schedule
* Compare results across agent versions to track safety improvements